# SmartCity AI — Business Classification Model
### متصل بـ SQL Server — الأسماء الحقيقية من الداتابيز

In [ ]:
# Cell 1 — تثبيت المتطلبات
import sys
!{sys.executable} -m pip install scikit-learn pandas numpy pyodbc sqlalchemy --quiet

In [ ]:
# Cell 2 — إعدادات الاتصال بـ SQL Server
DB_SERVER = r"NORASALMA\MSSQLSERVER01"   # ← اسم السيرفر من SSMS
DB_NAME   = "SmartCity"
DB_DRIVER = "ODBC Driver 17 for SQL Server"

print("✅ إعدادات الاتصال جاهزة")

In [ ]:
# Cell 3 — الاتصال بالداتابيز وقراءة البيانات
import pandas as pd
from sqlalchemy import create_engine

driver_encoded = DB_DRIVER.replace(' ', '+')
engine = create_engine(
    f"mssql+pyodbc://{DB_SERVER}/{DB_NAME}"
    f"?driver={driver_encoded}&trusted_connection=yes&TrustServerCertificate=yes"
)

print("✅ اتصال ناجح بـ SQL Server")

# ── Dim_Area ──
df_areas = pd.read_sql("""
    SELECT area_id, area_name, population, latitude, longitude
    FROM Dim_Area
""", engine)

# ── Dim_Business_Type ──
df_btypes = pd.read_sql("""
    SELECT business_type_id, category, subcategory, service_type
    FROM Dim_Business_Type
""", engine)

# ── Fact_Property_Suitability ──
df = pd.read_sql("""
    SELECT
        f.fact_id,
        f.prop_id,
        f.area_id,
        a.area_name,
        a.population,
        f.business_type_id       AS btype_id,
        f.category,
        f.street_name,
        f.area_sqm,
        f.rent_monthly_egp       AS rent_egp,
        f.rent_per_sqm,
        f.competitors_500m       AS comp_500m,
        f.competitors_1km        AS comp_1km,
        f.affordability_score    AS affordability,
        f.suitability_score      AS suitability,
        f.recommended
    FROM Fact_Property_Suitability f
    JOIN Dim_Area a ON f.area_id = a.area_id
""", engine)

print(f"✅ تم تحميل البيانات:")
print(f"   مناطق  : {len(df_areas)}")
print(f"   أنشطة  : {len(df_btypes)}")
print(f"   سجلات  : {len(df)}")
df.head()

In [ ]:
# Cell 4 — استكشاف البيانات
print("=" * 50)
print("توزيع الفئات:")
print(df["category"].value_counts())

print("\nإحصائيات الإيجار والملاءمة:")
print(df[["rent_egp", "rent_per_sqm", "suitability"]].describe().round(2))

print("\nأكثر 10 مناطق سجلات:")
print(df.groupby("area_name")["prop_id"].count().sort_values(ascending=False).head(10))

print("\nأنواع الأنشطة:")
print(df_btypes[["category","subcategory","service_type"]])

In [ ]:
# Cell 5 — تجهيز البيانات للتدريب
import numpy as np
import warnings
from sklearn.preprocessing import LabelEncoder, StandardScaler
warnings.filterwarnings("ignore")

# تصنيف درجة الملاءمة إلى 3 فئات
def tier(score):
    if score >= 7.5: return "High"
    if score >= 5.0: return "Medium"
    return "Low"

df["tier"] = df["suitability"].apply(tier)

# Encode الفئة
le_cat = LabelEncoder()
df["category_enc"] = le_cat.fit_transform(df["category"])

FEATURES = ["area_sqm", "rent_per_sqm", "affordability",
            "comp_500m", "comp_1km", "population", "category_enc"]

X       = df[FEATURES].values
y_tier  = df["tier"].values
y_score = df["suitability"].values

print(f"✅ Features : {FEATURES}")
print(f"   Samples  : {len(X)}")
print(f"\nTier distribution:")
print(df["tier"].value_counts())

In [ ]:
# Cell 6 — تدريب Model A: Tier Classifier (RandomForest)
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

le_tier    = LabelEncoder()
y_tier_enc = le_tier.fit_transform(y_tier)

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestClassifier(
        n_estimators=200, max_depth=8,
        min_samples_leaf=2, class_weight="balanced", random_state=42
    ))
])

cv        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y_tier_enc, cv=cv, scoring="accuracy")
clf.fit(X, y_tier_enc)

print(f"✅ Model A — Tier Classifier")
print(f"   CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

rf = clf.named_steps["rf"]
print("\nFeature Importance:")
for feat, imp in sorted(zip(FEATURES, rf.feature_importances_), key=lambda x: -x[1]):
    bar = "█" * int(imp * 40)
    print(f"  {feat:<20} {imp:.3f}  {bar}")

In [ ]:
# Cell 7 — تدريب Model B: Score Regressor (GradientBoosting)
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score

reg = Pipeline([
    ("scaler", StandardScaler()),
    ("gbr", GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05,
        max_depth=4, subsample=0.8, random_state=42
    ))
])

y_pred_cv = cross_val_predict(reg, X, y_score, cv=5)
mae       = mean_absolute_error(y_score, y_pred_cv)
r2        = r2_score(y_score, y_pred_cv)
reg.fit(X, y_score)

print(f"✅ Model B — Suitability Score Regressor")
print(f"   CV MAE : {mae:.3f}")
print(f"   CV R²  : {r2:.3f}")

In [ ]:
# Cell 8 — حفظ الموديل
import pickle, json

AREAS = {
    int(row["area_id"]): (row["area_name"], int(row["population"]))
    for _, row in df_areas.iterrows()
}
BUSINESS_TYPES = {
    int(row["business_type_id"]): (row["category"], row["subcategory"])
    for _, row in df_btypes.iterrows()
}

model_bundle = {
    "classifier":     clf,
    "regressor":      reg,
    "le_tier":        le_tier,
    "le_category":    le_cat,
    "features":       FEATURES,
    "areas":          AREAS,
    "business_types": BUSINESS_TYPES,
    "tier_labels":    list(le_tier.classes_),
    "categories":     list(le_cat.classes_),
    "training_data":  df,
    "metrics": {
        "classifier_cv_accuracy": float(cv_scores.mean()),
        "regressor_cv_mae":       float(mae),
        "regressor_cv_r2":        float(r2),
    }
}

with open("smartcity_model.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

meta = {
    "areas":          {str(k): {"name": v[0], "population": v[1]} for k, v in AREAS.items()},
    "business_types": {str(k): {"category": v[0], "subcategory": v[1]} for k, v in BUSINESS_TYPES.items()},
    "categories":     list(le_cat.classes_),
    "tier_labels":    list(le_tier.classes_),
    "metrics":        model_bundle["metrics"],
}
with open("smartcity_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("✅ smartcity_model.pkl — تم الحفظ")
print("✅ smartcity_meta.json — تم الحفظ")
print(f"\nأداء النموذج:")
print(f"  Classifier Accuracy : {cv_scores.mean():.1%}")
print(f"  Regressor MAE       : {mae:.3f}")
print(f"  Regressor R²        : {r2:.3f}")

In [ ]:
# Cell 9 — اختبار التنبؤ على عقار واحد
def predict(area_sqm, rent_egp, category, area_id=None, comp_500m=0, comp_1km=0):
    population    = AREAS.get(area_id, (None, 10000))[1] if area_id else 10000
    area_name_val = AREAS.get(area_id, ("غير محدد", 0))[0] if area_id else "غير محدد"
    rent_per_sqm  = rent_egp / area_sqm
    affordability = max(0.2, min(2.0, 1 - (rent_per_sqm / 450 - 1) * 0.5))
    cat_enc       = le_cat.transform([category])[0]
    x             = np.array([[area_sqm, rent_per_sqm, affordability,
                               comp_500m, comp_1km, population, cat_enc]])
    score         = float(reg.predict(x)[0])
    score         = round(min(10.0, max(1.0, score)), 2)
    tier_label    = le_tier.inverse_transform([clf.predict(x)[0]])[0]

    print(f"{'='*45}")
    print(f"  العقار  : {area_sqm}م² | إيجار {rent_egp:,} جنيه")
    print(f"  النشاط  : {category}")
    print(f"  المنطقة : {area_name_val}")
    print(f"  ─────────────────────────────────────────")
    print(f"  الدرجة  : {score}/10")
    print(f"  التصنيف : {tier_label}")
    print(f"  توصية   : {'✅ مناسب' if score >= 6 else '⚠️ راجع الاختيار'}")
    return score, tier_label

# ── جرّبي هنا ──
predict(
    area_sqm  = 120,
    rent_egp  = 30000,
    category  = "Food & Beverage",
    area_id   = 5,        # Attarin
    comp_500m = 1,
    comp_1km  = 3
)

In [ ]:
# Cell 10 — أفضل الأماكن لنشاط وميزانية معينة
def recommend_locations(category, max_rent, area_sqm, top_n=10):
    cat_enc = le_cat.transform([category])[0]
    results = []

    for area_id, (area_name_val, population) in AREAS.items():
        rows = df[(df["area_id"] == area_id) & (df["category"] == category)]

        if rows.empty:
            avg_rps  = min(1200, max(80, 450 * (population / 12000)))
            avg_c500 = 2
            avg_c1km = 5
        else:
            avg_rps  = rows["rent_per_sqm"].mean()
            avg_c500 = rows["comp_500m"].mean()
            avg_c1km = rows["comp_1km"].mean()

        est_rent = avg_rps * area_sqm
        if est_rent > max_rent:
            continue

        afford = max(0.2, min(2.0, 1 - (avg_rps / 450 - 1) * 0.5))
        x      = np.array([[area_sqm, avg_rps, afford,
                            avg_c500, avg_c1km, population, cat_enc]])

        score    = round(min(10.0, max(1.0, float(reg.predict(x)[0]))), 2)
        tier_lbl = le_tier.inverse_transform([clf.predict(x)[0]])[0]

        results.append({
            "منطقة":          area_name_val,
            "الدرجة":         score,
            "تصنيف":          tier_lbl,
            "إيجار تقديري":   int(est_rent),
            "منافسين 500م":   round(avg_c500, 1),
            "سكان":           population,
        })

    res_df = (pd.DataFrame(results)
                .sort_values("الدرجة", ascending=False)
                .head(top_n)
                .reset_index(drop=True))
    res_df.index += 1

    print(f"\n🏆 أفضل {top_n} مناطق لـ {category} | إيجار ≤ {max_rent:,} جنيه | {area_sqm}م²")
    return res_df

# ── جرّبي هنا ──
recommend_locations(
    category = "Retail",
    max_rent = 80000,
    area_sqm = 150
)

In [ ]:
# Cell 11 — تشغيل الـ API
# افتحي تيرمينال جديد في VS Code وشغّلي:
#
#   uvicorn smartcity_api:app --reload --port 8000
#
# ثم افتحي المتصفح على: http://localhost:8000/docs

print("لتشغيل الـ API، افتحي تيرمينال جديد وشغّلي:")
print()
print("  uvicorn smartcity_api:app --reload --port 8000")
print()
print("ثم افتحي: http://localhost:8000/docs")